# Session 3 Introduction: Maya's Learning Engine at Lindenfield

Six weeks after Dr. Sara Kim's Cobb-Douglas rebalancing engine went live on the Balanced pilot account at __Lindenfield Wealth Partners__, the engine has held its own through the April 2025 tariff correction and the summer recovery. The \$2M pilot is in the black, the drawdown trigger fired once in early April and restored capital through the rebound. Lou wants to scale the engine from the single pilot to the full __Balanced book__: eighty client accounts, roughly \$50M under management. 

Before the scale-up, three concerns sit on Maya's desk:
* __Parameter drift__. The engine's $\alpha$ and $\beta$ inputs were calibrated once, on 2014-2024 data. The AI-concentration rotation that dominated 2025 has shifted the cross-section of realized alphas; the frozen numbers no longer match what the market is paying today.
* __Hand-tuned elasticity__. Sara's adaptive CES elasticity $\eta(\lambda_t) = \eta_{\min} + (\eta_{\max} - \eta_{\min})/(1 + |\lambda_t|)$ has a reasonable shape but no theoretical justification beyond inspection. Lou will not sign a scale-up on a gut-feel formula, and compliance wants _data-driven_ behind the allocator, not _designed-by-inspection_.
* __No formal go/no-go__. The pilot has performed well, but performance does not equal approval. Compliance has requested pre-approved operating limits before the engine can trade the eighty client accounts without a human on every order.

__Scenario__: Thursday 1:30 PM, pre-deployment validation review. Friday 3:00 PM, scale-up sign-off with Lou and compliance. Maya needs three answers she can walk into each meeting with.

By the end of this session Maya will have three answers. First, does online re-estimation of the SIM parameters (EWLS) actually track parameter drift in real time, or is the frozen 2014-2024 calibration good enough for the scale-up? Second, does a multi-armed bandit with a separate instance per sentiment regime learn an $\eta$ map that matches or beats Sara's heuristic? Third, does the engine pass five formal deployment gates, and what operating limits (concentration cap, drawdown gate, turnover limit, position-size limit) export to Friday's compliance packet for Session 4's production system?

___

In [1]:
include("Include.jl"); # Load packages and helper functions. Activates the local Julia environment.

  Activating project at `~/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-3`
    Updating git-repo `https://github.com/varnerlab/alpaca-markets-sdk.git`
   Installed OpenBLAS32_jll ─ v0.3.32+0
   Installed JuMP ─────────── v1.30.1
  Installing 1 artifacts
   Installed artifact OpenBLAS32            8.0 MiB
    Updating `~/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-3/Project.toml`
  [c1a5797d] + Alpaca v0.2.0 `https://github.com/varnerlab/alpaca-markets-sdk.git#`
  [5ae59095] + Colors v0.13.1
  [a93c6f00] + DataFrames v1.8.2
  [09f84164] + HypothesisTests v0.11.7
  [91a5bcdd] + Plots v1.41.6
  [08abe8d2] + PrettyTables v3.3.2
  [f3b207a7] + StatsPlots v0.15.8
  [a1b2c3d4] + eCornellAIFinance v0.1.0 `../../code`
  [ade2ca70] ~ Dates ⇒ v1.11.0
    Updating `~/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-3/Manifest.toml`
  [14f7f29c] + AMD v0.5.3
  [621f4979] + AbstractFFTs v1.5.0
  [1520ce14] + AbstractTrees v0.4.5
  [7d9f7c33] +

## The Calibration Under Pressure
Let's start with the single number Lou will ask about on Thursday: the engine's frozen $\beta$ on a high-beta basket position. The plot below re-estimates that $\beta$ recursively on real 2025-2026 daily growth rates with a 63-trading-day half-life (EWLS). The dashed line is the 2014-2024 calibration value; the orange line is what the market has been telling the estimator as each day arrives. 

In [ ]:
let
    # --- Step 1: Load the frozen SIM calibration from Session 1 ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    sim_estimates = minvar["sim_estimates"];
    σ_m           = Float64(minvar["sigma_market"]);

    # --- Step 2: Load real 2025 + 2026 YTD daily OHLC (need the market index + at least one basket ticker) ---
    ds = MyExtendedTestingMarketDataSet()["dataset"]::Dict{String,DataFrame};
    haskey(ds, "SPY") || error("SPY missing from MyExtendedTestingMarketDataSet; intro cannot plot β drift.");

    # --- Step 3: Choose focus ticker; prefer NVDA if in the basket and real-data loader, otherwise fall back to the highest-β basket ticker with real coverage ---
    preferred  = "NVDA";
    candidates = [e for e in sim_estimates if haskey(ds, e.ticker)];
    isempty(candidates) && error("no basket ticker has real 2025-2026 OHLC coverage; intro cannot plot β drift.");
    k_pref = findfirst(e -> e.ticker == preferred, candidates);
    chosen = k_pref !== nothing ? candidates[k_pref] :
             candidates[argmax([Float64(e.β) for e in candidates])];
    focus_ticker = chosen.ticker;
    β_frozen = Float64(chosen.β);
    α_frozen = Float64(chosen.α);
    if focus_ticker != preferred
        println("NVDA not in the user's basket or real-data loader; falling back to $(focus_ticker) (highest-β available, β_frozen = $(round(β_frozen, digits=3))).");
    end
    df_i = ds[focus_ticker];
    df_m = ds["SPY"];

    # --- Step 4: Compute daily annualized continuously compounded growth rates ---
    # g_t = (1/Δt) · log(S_t / S_{t-1}) with Δt = 1/252
    Δt = 1.0 / 252.0;
    n = min(size(df_i, 1), size(df_m, 1)) - 1;
    g_i = [(1.0 / Δt) * log(df_i.close[t + 1] / df_i.close[t]) for t ∈ 1:n];
    g_m = [(1.0 / Δt) * log(df_m.close[t + 1] / df_m.close[t]) for t ∈ 1:n];

    # --- Step 5: EWLS recursion with 63-day half-life, prior seeded from frozen calibration ---
    # δ is the decay factor; each step ages every sufficient statistic by δ before adding today's contribution.
    h  = 63;
    δ  = 2.0 ^ (-1.0 / h);
    W0 = 84;                                       # prior anchor strength (observation-equivalents)
    Sw   = Float64(W0);
    Swx  = 0.0;                                    # E[g_m] ≈ 0 prior
    Swy  = W0 * α_frozen;                          # E[g_i] ≈ α_frozen prior
    Swxx = W0 * σ_m ^ 2;                           # Var[g_m] ≈ σ_m² prior
    Swxy = W0 * β_frozen * σ_m ^ 2;                # Cov[g_i, g_m] ≈ β · σ_m² prior
    β_series = zeros(n);
    for t ∈ 1:n
        Sw   = δ * Sw   + 1.0;
        Swx  = δ * Swx  + g_m[t];
        Swy  = δ * Swy  + g_i[t];
        Swxx = δ * Swxx + g_m[t] ^ 2;
        Swxy = δ * Swxy + g_i[t] * g_m[t];
        # β̂ = (S_w · S_wxy − S_wx · S_wy) / (S_w · S_wxx − S_wx²)
        det_ = Sw * Swxx - Swx ^ 2;
        β_series[t] = det_ > 1e-10 ? (Sw * Swxy - Swx * Swy) / det_ : β_frozen;
    end

    # --- Step 6: Plot frozen vs. EWLS-rolling β ---
    days = 1:n;
    pl = plot(days, β_series;
        lw = 2.5, c = :darkorange,
        label = "EWLS β (h = 63 days)",
        xlabel = "Trading day (2025-01 through 2026 YTD)", ylabel = "β",
        size = (800, 450), fontsize = 14, left_margin = 8Plots.mm);
    hline!(pl, [β_frozen]; lw = 2.5, ls = :dash, c = :gray20,
        label = "Frozen β from S1 (2014-2024) = $(round(β_frozen, digits=3))");
    plot!(pl, bg = "gray95", background_color_outside = "white",
        framestyle = :box, fg_legend = :transparent, legend = :bottomright);
    title!(pl, "$(focus_ticker) β: frozen 2014-2024 vs. online EWLS on real 2025-2026 data");
    display(pl);
end;

One ticker's $\beta$ is not a full argument, but it makes the first point: if the EWLS line drifts noticeably from the frozen line, the engine's preference-weight formula has been reading stale numbers. __The fix is online re-estimation__.

Second, compliance has proposed five pass/fail gates for production scale-up: median Sharpe at least 0.3, median max drawdown at most 25%, failure rate (paths ending below starting capital) at most 10%, the engine's median terminal wealth must beat the S1 min-var passive baseline, and the engine's median net present value must be non-negative against the risk-free rate. Below is where the current Session 2 engine (frozen SIM + hand-tuned $\eta$) sits against each gate across the Monte Carlo path ensemble from last session.

In [ ]:
let
    # --- Step 1: Load the Session 2 engine path ensemble ---
    st      = load_results(joinpath(_PATH_TO_DATA_S2, "stress-test-engine.jld2"));
    eng_fw  = st["eng_final_wealth"]::Vector{Float64};
    eng_dd  = st["eng_max_drawdowns"]::Vector{Float64};
    eng_sr  = st["eng_sharpe_ratios"]::Vector{Float64};
    mv_fw   = st["mv_final_wealth"]::Vector{Float64};
    B₀      = Float64(st["B₀"]);
    n_paths = length(eng_fw);

    # --- Step 2: Pull g_f from S1 so the NPV gate uses the course-wide risk-free rate ---
    g_f = Float64(load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"))["g_f"]);
    T_active = 1.0;                                 # one trading year of active engine time
    discount = exp(-g_f * T_active);

    # --- Step 3: Compute the five deployment-gate metrics ---
    med_sharpe = median(eng_sr);
    med_dd     = median(eng_dd);
    fail_rate  = sum(eng_fw .< B₀) / n_paths;
    bh_ratio   = median(eng_fw) / median(mv_fw);
    med_npv    = median(eng_fw .* discount .- B₀);

    # --- Step 4: Assemble a pass/fail row per gate ---
    rows = [
        (Gate = "Median Sharpe",            Direction = "≥", Threshold = "0.30",
         Actual = "$(round(med_sharpe, digits=3))",
         Verdict = med_sharpe >= 0.30 ? "✓ PASS" : "✗ FAIL"),
        (Gate = "Median Max Drawdown",      Direction = "≤", Threshold = "25%",
         Actual = "$(round(med_dd * 100, digits=1))%",
         Verdict = med_dd <= 0.25 ? "✓ PASS" : "✗ FAIL"),
        (Gate = "Failure Rate",             Direction = "≤", Threshold = "10%",
         Actual = "$(round(fail_rate * 100, digits=1))%",
         Verdict = fail_rate <= 0.10 ? "✓ PASS" : "✗ FAIL"),
        (Gate = "Wealth vs Min-Var",        Direction = "≥", Threshold = "1.00",
         Actual = "$(round(bh_ratio, digits=3))",
         Verdict = bh_ratio >= 1.0 ? "✓ PASS" : "✗ FAIL"),
        (Gate = "Median NPV",               Direction = "≥", Threshold = "\$0",
         Actual = "\$$(round(med_npv, digits=0))",
         Verdict = med_npv >= 0.0 ? "✓ PASS" : "✗ FAIL"),
    ];
    df = DataFrame(rows);

    # --- Step 5: Render the gate summary Maya will walk into Thursday with ---
    println("Thursday validation preview: S2 engine (frozen SIM + hand-tuned η) on the Monte Carlo path ensemble")
    println("Engine config: DD ≤ 15%, τ ≤ 50%, daily rebalance, bias-corrected to 8%/yr prior")
    println()
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end;

## Is Maya Ready for Friday?
The online $\beta$ does not stay glued to the frozen 2014-2024 value, and the synthetic-ensemble preview clears all five gates with comfortable margins. Both observations make the same point: the engine looks healthy on paper, but Lou will not sign a scale-up on a synthetic-ensemble win. The Sharpe of 2.7 inherits the SIM rebalancing-alpha artifact that Session 2 already documented; the $\beta$ drift on real 2025-2026 data is exactly the kind of shift the synthetic ensemble cannot reproduce. Three upgrades have to be operational before the Friday packet:

* __Upgrade 1, online SIM.__ Sara's fix is EWLS: re-estimate $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ daily with an exponential-decay weighting so the preference-weight formula always reads fresh parameters. The lecture derives the recursion and its half-life and prior-seeding knobs, and the first Core example replays the engine on the Monte Carlo ensemble with frozen vs. online inputs side-by-side.
* __Upgrade 2, learned $\eta$.__ Sara replaces the hand-tuned $\eta(\lambda_t)$ heuristic with an epsilon-greedy multi-armed bandit, one instance per sentiment regime (bearish, neutral, bullish), over a discrete $\eta$ grid. The lecture covers the regret framing and the exploration-decay schedule; the second Core example runs the bandit on the ensemble and compares the learned $\eta$ map to Sara's heuristic head-to-head.
* __Upgrade 3, formal validation.__ The third Core example assembles the five-gate validation report and exports the compliance configuration (concentration cap, drawdown gate, turnover limit, position-size limit) that Session 4's production system enforces in real time. Friday's sign-off packet is the JLD2 that this example writes to disk.

Maya cannot walk into Friday yet. The lecture that follows develops each upgrade formally; the core examples that follow produce the numbers Maya needs to make the case.
___